In [ ]:
import os
# Blackwell safety: disable torch.compile / dynamo paths that can fail to launch on sm_120.
# MUST be set before importing unsloth.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ.setdefault("TORCHDYNAMO_DISABLE", "1")
os.environ.setdefault("UNSLOTH_COMPILE_DISABLE", "1")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
# Unsloth must be imported BEFORE transformers / trl / peft so its patches apply.
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only

import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

print("torch", torch.__version__, "| cuda", torch.version.cuda,
      "| capability", torch.cuda.get_device_capability())

MODEL_KEY = "qwen05"   

MODELS = {
    "qwen05": dict(
        name="unsloth/Qwen2.5-Coder-0.5B-Instruct",
        load_in_4bit=False,
        instr_part="<|im_start|>user\n",
        resp_part="<|im_start|>assistant\n",
    ),
    "gemma3": dict(
        name="unsloth/gemma-3-1b-it",
        load_in_4bit=False,
        instr_part="<start_of_turn>user\n",
        resp_part="<start_of_turn>model\n",
    ),
}

cfg = MODELS[MODEL_KEY]

# final files produced by the cleaning step
TRAIN_FILE   = "../data/cleaned/train_set_finetuned.jsonl"
VAL_FILE     = "../data/cleaned/test_set_150_finetuned.jsonl"
OUTDIR       = f"out_{MODEL_KEY}_fullstrip"
MAX_SEQ_LEN  = 512
EPOCHS       = 50       
LR           = 2e-4
BATCH_SIZE   = 4
GRAD_ACCUM   = 4
SEED         = 42

print("Model:", cfg["name"], "| 4bit:", cfg["load_in_4bit"], "| out:", OUTDIR)

In [ ]:
# Load the model only to get the tokenizer for formatting the data; it is released afterwards
# (run_training_for_rank loads a fresh model for every rank)
_load_kwargs = dict(
    model_name=cfg["name"],
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=False,
    dtype=torch.bfloat16,
)
if "gemma" in cfg["name"].lower():
    _load_kwargs["text_only"] = True
_tmp_model, tokenizer = FastLanguageModel.from_pretrained(**_load_kwargs)

def fmt(ex):
    msgs = ex["messages"]
    # Gemma does not accept a system role; fold the system text into the first user turn
    if "gemma" in cfg["name"].lower():
        sys_c = ""
        for m in msgs:
            if m["role"] == "system":
                sys_c = m["content"]
                break
        new_msgs = []
        first_user_done = False
        for m in msgs:
            if m["role"] == "system":
                continue
            if m["role"] == "user" and not first_user_done:
                if sys_c:
                    merged = sys_c + "\n\n" + m["content"]
                else:
                    merged = m["content"]
                new_msgs.append({"role": "user", "content": merged})
                first_user_done = True
            else:
                new_msgs.append(m)
        msgs = new_msgs
    text = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=False)
    return {"text": text}

train_ds = load_dataset("json", data_files=TRAIN_FILE, split="train").map(fmt)
val_ds_raw = load_dataset("json", data_files=VAL_FILE, split="train")   # raw version, for CompositeEvalCallback
val_ds     = val_ds_raw.map(fmt)                                          # formatted version, for SFTTrainer's own eval_dataset
print("train:", len(train_ds), "| val:", len(val_ds))
print("\n--- sample formatted text ---\n", train_ds[0]["text"][:500])

# Release the temporary model; keep only the tokenizer and the formatted datasets
del _tmp_model
torch.cuda.empty_cache()

In [ ]:
from training_callback import CompositeEvalCallback
from transformers import EarlyStoppingCallback

def run_training_for_rank(rank, train_seed, lora_alpha=None, epochs=EPOCHS, ref_manifest_path=None):
    """One full run: load a fresh base model -> attach LoRA at the given rank -> train -> save the adapter.
    The base model is reloaded every time (PEFT attachment is stateful; one model object cannot be re-attached at another rank).
    train_seed controls the randomness of training (data order, LoRA A-matrix initialisation, dropout masks);
    it separates a real rank effect from training noise."""
    if lora_alpha is None:
        lora_alpha = rank  # convention used throughout: alpha == r

    # the seed is part of the directory name so runs of the same rank do not overwrite each other
    rank_outdir = f"out_{MODEL_KEY}_r{rank}_seed{train_seed}_fullstrip"
    print(f"\n{'='*60}\nTraining: rank={rank} seed={train_seed} lora_alpha={lora_alpha} -> {rank_outdir}\n{'='*60}")

    # 1. Load a fresh base model and tokenizer
    load_kwargs = dict(
        model_name=cfg["name"],
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False,
        dtype=torch.bfloat16,
    )
    # By default unsloth loads Gemma 3 through its vision path (vision.py) with device_map="auto",
    # which cannot be used for training. text_only=True forces the text-only path.
    if "gemma-3" in cfg["name"].lower():
        load_kwargs["text_only"] = True

    base_model, tok = FastLanguageModel.from_pretrained(**load_kwargs)

    # 2. Attach LoRA at this rank; random_state = train_seed (initialisation of the LoRA A matrix)
    peft_model = FastLanguageModel.get_peft_model(
        base_model,
        r=rank,
        lora_alpha=lora_alpha,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=train_seed,          # <-- train_seed (previously a fixed SEED)
        use_rslora=False,
        loftq_config=None,
    )

    # 3. Composite callback (including Onset F1 / Hap F1), writing to this rank+seed's output directory
    composite_cb = CompositeEvalCallback(
        val_raw=val_ds_raw,
        tokenizer=tok,
        model_name=cfg["name"],
        outdir=rank_outdir,
        ref_manifest_path=ref_manifest_path,
    )

    # 4. Trainer; seed = train_seed (data order, dropout masks)
    trainer = SFTTrainer(
        model=peft_model,
        tokenizer=tok,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        args=SFTConfig(
            output_dir=rank_outdir,
            dataset_text_field="text",
            max_seq_length=MAX_SEQ_LEN,
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM,
            warmup_ratio=0.05,
            num_train_epochs=epochs,
            learning_rate=LR,
            logging_steps=10,
            eval_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=None,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="cosine",
            seed=train_seed,              # <-- train_seed (previously a fixed SEED)
            bf16=True,
            report_to="none",
            metric_for_best_model="full_composite",
            greater_is_better=True,
            load_best_model_at_end=True,
        ),
        callbacks=[
            composite_cb,
            EarlyStoppingCallback(
                early_stopping_patience=3,
                early_stopping_threshold=1e-5,
            ),
        ],
    )

    trainer = train_on_responses_only(
        trainer,
        instruction_part=cfg["instr_part"],
        response_part=cfg["resp_part"],
    )

    trainer.train()
    best_metric = trainer.state.best_metric
    best_checkpoint = trainer.state.best_model_checkpoint

    peft_model.save_pretrained(f"{rank_outdir}/adapter")
    tok.save_pretrained(f"{rank_outdir}/adapter")
    print(f"rank={rank} seed={train_seed}: adapter saved to {rank_outdir}/adapter, "
          f"best_metric={best_metric}, best_checkpoint={best_checkpoint}")

    del base_model, peft_model, trainer
    torch.cuda.empty_cache()

    return {"rank": rank, "train_seed": train_seed, "lora_alpha": lora_alpha,
            "outdir": rank_outdir, "best_metric": best_metric,
            "best_checkpoint": best_checkpoint}

In [ ]:
import json
import glob

RANKS_TO_COMPARE = [224]  # thesis sweeps: qwen05 16,48,64,80,112 | gemma3 16,64,128,192,224
TRAIN_SEEDS = [42, 123, 2024]  # the three training seeds of the thesis
REF_MANIFEST_PATH = "audio_capture/reference/test_set_150_finetuned/reference_manifest.jsonl"

rank_results = []
for rank in RANKS_TO_COMPARE:
    for train_seed in TRAIN_SEEDS:
        rank_results.append(
            run_training_for_rank(rank, train_seed, ref_manifest_path=REF_MANIFEST_PATH)
        )
        # Write results after every run so a crash does not lose the earlier ones
        with open("rank_seed_sweep_results.json", "w") as f:
            json.dump(rank_results, f, indent=2)

# Summary: full metrics of the best epoch for each (rank, seed)
print("\n" + "="*90)
print(f"{'Rank':<6}{'Seed':<8}{'Best Ckpt':<16}{'FullComp':<12}{'Syntax':<9}{'Exec':<9}{'Text':<9}{'Pearson':<9}{'SSIM':<9}{'Onset':<9}{'Hap':<9}")
print("="*90)
for r in rank_results:
    log_path = os.path.join(r["outdir"], "epoch_eval", "epoch_ranking_log.jsonl")
    best_row = None
    if os.path.isfile(log_path):
        with open(log_path, "r", encoding="utf-8") as f:
            rows = [json.loads(l) for l in f if l.strip()]
        matches = [row for row in rows if abs(row["full_composite"] - r["best_metric"]) < 1e-6]
        best_row = matches[0] if matches else max(rows, key=lambda x: x["full_composite"])
    if best_row:
        print(f"{r['rank']:<6}{r['train_seed']:<8}{os.path.basename(r['best_checkpoint'] or ''):<16}"
              f"{best_row['full_composite']:<12.4f}{best_row['syntax_pass_rate']:<9.4f}"
              f"{best_row['execution_pass_rate']:<9.4f}{best_row['text_similarity']:<9.4f}"
              f"{best_row['mel_pearson']:<9.4f}{best_row['mel_ssim']:<9.4f}"
              f"{best_row['onset_f1']:<9.4f}{best_row.get('hap_f1', 0):<9.4f}")
    else:
        print(f"{r['rank']:<6}{r['train_seed']:<8} (no ranking log found)")